# ReelForge — One-Prompt AI Video Studio

**Made for phone-first use. You do not need to write or edit code.**

1. Tap **Connect** in the top-right corner.
2. Open **Runtime → Run all**. Quick Cut works without a GPU; choose an available GPU for Fast AI, Balanced or High detail.
3. When ReelForge opens, upload your raw clips/photos, optionally add your own audio, write one prompt, then tap **Create my video**.
4. Choose **Quick Cut** to edit your uploaded clips without downloading an AI video model. Choose **Fast AI** for short AI-generated scenes, or **Balanced / High detail** for fuller AI scene coverage. Quick Cut is the default and the starting length is 15 seconds. The notebook installs only the basic editor packages at launch; AI packages install only when you choose an AI render mode.

ReelForge uses open model weights instead of paid generation APIs. Wan 2.1 is the primary text-to-video engine; AnimateDiff is the moving-video fallback. The renderer does **not** silently replace failed video generation with still images.

The finished export is a vertical H.264/AAC MP4, with no added watermark. The pipeline can analyse uploaded frames, create AI B-roll, edit source media, add transitions/VFX, transcribe speech, generate a voice-over when appropriate, create an original instrumental score when no audio is supplied, and add transition sound effects.

**Speed modes:** Quick Cut uses your uploaded videos/photos directly, skips BLIP, Qwen and Whisper model downloads, and can run without a GPU. Fast AI uses Wan at 33 frames, 8 fps and 4 steps, skips the separate BLIP/Qwen director downloads, and stops rather than downloading a second slow video model if Wan fails. Balanced uses 65 frames at 12 steps; High detail uses 81 frames at 16 steps. The Wan model stays loaded during the current Colab session, so a second render skips reloading it. The progress card reports scene and diffusion-step progress while a shot is being generated.

The notebook runs in Google's Colab cloud runtime. Your media is uploaded to that runtime for processing; this repository stores the code, not your uploaded media. Free GPU access and session availability are controlled by Colab and are not guaranteed.

Use media and music you own or have permission to use. AI scenes are short generated shots assembled into a complete edit, not a single continuous one-minute generated shot.

In [ ]:
#@title Install ReelForge's quick-edit core (AI models install only when selected)
import importlib.metadata as metadata
import subprocess
import sys
import shutil
import os
from packaging.specifiers import SpecifierSet
from packaging.version import Version

# Quick Cut only needs the editor, media codecs and audio utilities.
apt_packages = []
if not shutil.which("ffmpeg"):
    apt_packages.append("ffmpeg")
if not shutil.which("espeak-ng"):
    apt_packages.append("espeak-ng")
if not os.path.exists("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"):
    apt_packages.append("fonts-dejavu-core")
if not os.path.exists("/usr/share/fonts/truetype/noto/NotoSansDevanagari-Regular.ttf"):
    apt_packages.append("fonts-noto-core")

if apt_packages:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", *apt_packages], check=True)

requirements = [
    ("numpy", "numpy<2", "<2"),
    ("moviepy", "moviepy==1.0.3", "==1.0.3"),
    ("imageio", "imageio[ffmpeg]>=2.34", ">=2.34"),
    ("imageio-ffmpeg", "imageio-ffmpeg", ""),
    ("soundfile", "soundfile>=0.12.1", ">=0.12.1"),
    ("ipywidgets", "ipywidgets>=8.0", ">=8.0"),
    ("pillow", "Pillow>=9.0", ">=9.0"),
]

to_install = []
for distribution, requirement, version_rule in requirements:
    try:
        installed = Version(metadata.version(distribution))
        if version_rule and installed not in SpecifierSet(version_rule):
            to_install.append(requirement)
    except metadata.PackageNotFoundError:
        to_install.append(requirement)

if to_install:
    print("Installing quick-edit packages:", ", ".join(to_install))
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *to_install],
        check=True
    )
else:
    print("Quick-edit packages already match. Skipping pip installs.")

print("Quick Cut is ready. AI model packages will install only if you select Fast AI / Balanced / High detail.")


In [ ]:
#@title Check available rendering hardware
import torch
from PIL import Image

if not hasattr(Image, "ANTIALIAS") and hasattr(Image, "Resampling"):
    Image.ANTIALIAS = Image.Resampling.LANCZOS

if torch.cuda.is_available():
    print("AI video GPU ready:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")
else:
    print("No GPU attached. Quick Cut can still edit uploaded footage. Fast AI / Balanced / High detail need an available GPU.")


In [ ]:
#@title Launch ReelForge Studio
import urllib.request

REELFORGE_SCRIPT = "https://raw.githubusercontent.com/manjunathjadav360-rgb/Reel-forge-ai/main/colab/reelforge_cloud.py"
with urllib.request.urlopen(REELFORGE_SCRIPT, timeout=60) as response:
    source = response.read().decode("utf-8")

exec(compile(source, "reelforge_cloud.py", "exec"), globals())